# Preprocessing and feature engineering

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
from IPython.display import display, Image, Markdown
from src.data_loader import load_partition, predictors, fingerprints
def report(name):
    display(Markdown((ROOT / 'reports' / name).read_text(encoding='utf-8')))
def figure(name):
    display(Image(filename=str(ROOT / 'figures' / name)))


In [2]:
from src.features import NetworkFeatures
train=pd.read_csv(ROOT/'data/processed/train.csv')
validation=pd.read_csv(ROOT/'data/processed/validation.csv')
test=pd.read_csv(ROOT/'data/processed/test.csv')
assert not set(fingerprints(train)) & set(fingerprints(validation))
assert not set(fingerprints(pd.concat([train,validation]))) & set(fingerprints(test))
X=predictors(train)
assert not {'label','attack_cat','id'} & set(X.columns)
display(NetworkFeatures().fit_transform(X.head()))

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,total_bytes,total_packets,bytes_per_packet,source_byte_share,source_packet_share,bytes_per_second
0,0.000009,udp,-,INT,2,0,108,0,111111.107200,254,0,4.800000e+07,0.000000e+00,0,0,0.009000,0.000000,0.000000,0.000000,0,0,0,0,0.000000,0.000000,0.000000,54,0,0,0,26,2,12,12,1,26,0,0,0,12,26,0,108,2,54.000000,1.000000,1.000000,1.200000e+07
1,0.909934,tcp,-,FIN,10,6,558,268,16.484712,254,252,4.422299e+03,1.969374e+03,2,1,94.229111,143.422797,5099.302874,201.556578,255,696754686,3979343970,255,0.262688,0.192814,0.069874,56,45,0,0,6,1,2,1,1,3,0,0,0,1,3,0,826,16,51.625000,0.675545,0.625000,9.077581e+02
2,0.047962,tcp,-,FIN,52,54,3182,39076,2189.233201,31,29,5.205788e+05,6.397232e+06,7,20,0.952820,0.900442,58.389281,56.501523,255,4118648415,4114714815,255,0.001273,0.001138,0.000135,61,724,0,0,5,0,6,1,1,1,0,0,0,2,6,0,42258,106,398.660377,0.075299,0.490566,8.810725e+05
3,0.003864,tcp,-,FIN,12,12,1064,2260,5952.380820,31,29,2.020704e+06,4.289855e+06,4,4,0.320636,0.301000,16.211326,0.368254,255,2267528540,118194728,255,0.000694,0.000550,0.000144,89,188,0,0,1,0,3,1,1,1,0,0,0,5,5,0,3324,24,138.500000,0.320096,0.500000,8.602484e+05
4,0.000007,udp,-,INT,2,0,104,0,142857.140900,254,0,5.942857e+07,0.000000e+00,0,0,0.007000,0.000000,0.000000,0.000000,0,0,0,0,0.000000,0.000000,0.000000,52,0,0,0,5,2,2,2,1,5,0,0,0,2,5,0,104,2,52.000000,1.000000,1.000000,1.485714e+07


In [3]:
import joblib
model=joblib.load(ROOT/'models/final_model.joblib')
transformed=model[:-1].transform(X.head(100))
assert np.isfinite(transformed).all()
print('Original predictors:',X.shape[1], 'Transformed columns:',transformed.shape[1])
display(pd.read_csv(ROOT/'reports/feature_ablation.csv'))

Original predictors: 42 Transformed columns: 71


,features,n,accuracy,precision,recall,f1,roc_auc,pr_auc,fpr,fnr,tn,fp,fn,tp
0,Original,20163,0.933839,0.918028,0.949034,0.933273,0.986149,0.984248,0.080616,0.050966,9500,833,501,9329
1,Engineered,20163,0.934782,0.920079,0.948627,0.934135,0.986236,0.984289,0.078390,0.051373,9523,810,505,9325


Interpretation: median/mode/scaling/encoding parameters come from training. Ratios with zero denominators remain undefined until imputation. The validation ablation keeps hyperparameters fixed and does not reopen test-based model selection.